# CELL 0 — INSTALL / IMPORTS

In [ ]:
# ============================================================
# CELL 0 — INSTALL / IMPORTS
# ============================================================

!pip -q install lpips

import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# CELL 1 — NEW EXPERIMENT DIRECTORY

In [ ]:
# ============================================================
# CELL 1 — NEW EXPERIMENT DIRECTORY
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion"
)

DATA_DIR = BASE_DIR / "data"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
LOG_DIR = BASE_DIR / "logs"

for d in [BASE_DIR, DATA_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Experiment directory:")
print(BASE_DIR)

print("\nCheckpoints:")
print(CHECKPOINT_DIR)

print("\nLogs:")
print(LOG_DIR)

Mounted at /content/drive
Experiment directory:
/content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion

Checkpoints:
/content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/checkpoints

Logs:
/content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/logs


# CELL 2 — REPRODUCIBILITY

In [ ]:
# ============================================================
# CELL 2 — REPRODUCIBILITY
# ============================================================

SEED = 20260816

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("Seed:", SEED)

Seed: 20260816


# CELL 3 — DOWNLOAD CELEBA

In [ ]:
# ============================================================
# CELL 3 — DOWNLOAD CELEBA TO LOCAL VM DISK
# ============================================================

# Use local Colab VM disk (/content/celeba) instead of Drive for fast unzipping
CELEBA_ROOT = Path("/content/celeba")

print("CelebA root (Local VM):")
print(CELEBA_ROOT)

celeba_train = datasets.CelebA(
    root=str(CELEBA_ROOT),
    split="train",
    target_type="attr",
    download=True,
)

print("\nCelebA downloaded and unzipped successfully.")
print("Training images:", len(celeba_train))

print("Number of attributes:", len(celeba_train.attr_names))

print("\nSmiling attribute index:",
      celeba_train.attr_names.index("Smiling"))

CelebA root (Local VM):
/content/celeba


Downloading...
From (original): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM
From (redirected): https://drive.usercontent.google.com/download?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM&confirm=t&uuid=62d47799-ff77-4275-9009-c00c19fde490
To: /content/celeba/celeba/img_align_celeba.zip
100%|██████████| 1.44G/1.44G [00:12<00:00, 114MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: /content/celeba/celeba/list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:00<00:00, 132MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS
To: /content/celeba/celeba/identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:00<00:00, 46.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pbThiMVRxWXZ4dU0
To: /content/celeba/celeba/list_bbox_celeba.txt
100%|██████████| 6.08M/6.08M [00:00<00:00, 118MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pd0FJY3Blby1HUTQ
To: /content/celeba/celeba/li


CelebA downloaded and unzipped successfully.
Training images: 162770
Number of attributes: 41

Smiling attribute index: 31


# CELL 4 — PREPROCESSING

In [ ]:
# ============================================================
# CELL 4 — PREPROCESSING
# ============================================================

IMAGE_SIZE = 256

train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

print("Victim input shape:")
print(f"(3, {IMAGE_SIZE}, {IMAGE_SIZE})")

print("Flattened dimension:")
print(3 * IMAGE_SIZE * IMAGE_SIZE)

Victim input shape:
(3, 256, 256)
Flattened dimension:
196608


# CELL 5 — SMILING DATASET

In [ ]:
# ============================================================
# CELL 5 — SMILING DATASET
# ============================================================

class CelebASmiling(torch.utils.data.Dataset):

    def __init__(self, root, transform=None):
        self.dataset = datasets.CelebA(
            root=str(root),
            split="train",
            target_type="attr",
            transform=transform,
            download=True,
        )

        self.smiling_idx = self.dataset.attr_names.index("Smiling")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, attrs = self.dataset[idx]

        # CelebA encodes attributes as -1 / +1.
        label = int(attrs[self.smiling_idx].item())

        # Convert:
        # -1 -> 0
        # +1 -> 1
        label = 1 if label == 1 else 0

        return image, label


train_dataset = CelebASmiling(
    CELEBA_ROOT,
    transform=train_transform,
)

print("Training samples:", len(train_dataset))

x0, y0 = train_dataset[0]

print("Example image shape:", tuple(x0.shape))
print("Example label:", y0)

assert x0.shape == (3, 256, 256)
assert y0 in [0, 1]


Training samples: 162770
Example image shape: (3, 256, 256)
Example label: 1


# CELL 6 — DATALOADER

In [ ]:
# ============================================================
# CELL 6 — DATALOADER
# ============================================================

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(train_loader))

Batch size: 128
Number of batches: 1272


# CELL 7 — SMALL MLP

In [ ]:
# ============================================================
# CELL 7 — SMALL MLP
# ============================================================

class SmallMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(
            3 * 256 * 256,
            128,
        )

        self.act1 = nn.Sigmoid()

        self.fc2 = nn.Linear(
            128,
            2,
        )

    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        x = self.act1(self.fc1(x))
        x = self.fc2(x)
        return x


model = SmallMLP()

print(model)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

SmallMLP(
  (fc1): Linear(in_features=196608, out_features=128, bias=True)
  (act1): Sigmoid()
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)
Parameters: 25166210


# CELL 8 — LENET-STYLE CNN

In [ ]:
# ============================================================
# CELL 8 — LENET-STYLE CNN
# ============================================================

class LeNetCelebA(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3, 12,
                kernel_size=5,
                stride=2,
                padding=2,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                12, 12,
                kernel_size=5,
                stride=2,
                padding=2,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                12, 12,
                kernel_size=5,
                stride=2,
                padding=2,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                12, 12,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.Sigmoid(),
        )

        # 256 -> 128 -> 64 -> 32
        self.fc = nn.Linear(
            12 * 32 * 32,
            2,
        )

    def forward(self, x):
        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x


model = LeNetCelebA()

print(model)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

LeNetCelebA(
  (features): Sequential(
    (0): Conv2d(3, 12, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (1): Sigmoid()
    (2): Conv2d(12, 12, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (3): Sigmoid()
    (4): Conv2d(12, 12, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
    (5): Sigmoid()
    (6): Conv2d(12, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): Sigmoid()
  )
  (fc): Linear(in_features=12288, out_features=2, bias=True)
)
Parameters: 34022


# CELL 9 — GRADIENT-INVERSION-FRIENDLY CNN

In [ ]:
# ============================================================
# CELL 9 — GRADIENT-INVERSION-FRIENDLY CNN
# ============================================================

class GradientFriendlyCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3, 32,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                32, 32,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                64, 64,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.Sigmoid(),
        )

        # 256 -> 128 -> 64 -> 32 -> 16
        # Final spatial map: 16 x 16
        self.fc1 = nn.Linear(
            128 * 16 * 16,
            128,
        )

        self.act = nn.Sigmoid()

        self.fc2 = nn.Linear(
            128,
            2,
        )

    def forward(self, x):
        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x


model = GradientFriendlyCNN()

print(model)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

GradientFriendlyCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Sigmoid()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Sigmoid()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): Sigmoid()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): Sigmoid()
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): Sigmoid()
  )
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (act): Sigmoid()
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)
Parameters: 4334114


# CELL 10 — OVERPARAMETERIZED CNN

In [ ]:
# ============================================================
# CELL 10 — OVERPARAMETERIZED CNN
# ============================================================

class OverparameterizedCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3, 64,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                128, 256,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                256, 256,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
            nn.Sigmoid(),

            nn.Conv2d(
                256, 256,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.Sigmoid(),
        )

        # 256 -> 128 -> 64 -> 32 -> 16
        self.fc1 = nn.Linear(
            256 * 16 * 16,
            512,
        )

        self.act = nn.Sigmoid()

        self.fc2 = nn.Linear(
            512,
            2,
        )

    def forward(self, x):
        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x


model = OverparameterizedCNN()

print(model)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

OverparameterizedCNN(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): Sigmoid()
    (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): Sigmoid()
    (4): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): Sigmoid()
    (6): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): Sigmoid()
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): Sigmoid()
  )
  (fc1): Linear(in_features=65536, out_features=512, bias=True)
  (act): Sigmoid()
  (fc2): Linear(in_features=512, out_features=2, bias=True)
)
Parameters: 35106946


# CELL 11 — MODEL REGISTRY

In [ ]:
# ============================================================
# CELL 11 — MODEL REGISTRY
# ============================================================

MODEL_FACTORIES = {
    "mlp_small": SmallMLP,
    "lenet_sigmoid": LeNetCelebA,
    "cnn_gi": GradientFriendlyCNN,
    "cnn_overparam": OverparameterizedCNN,
}

print("Models to train:")

for name, factory in MODEL_FACTORIES.items():

    m = factory()

    n_params = sum(
        p.numel()
        for p in m.parameters()
    )

    print(
        f"{name:20s} "
        f"{n_params:,} parameters"
    )

Models to train:
mlp_small            25,166,210 parameters
lenet_sigmoid        34,022 parameters
cnn_gi               4,334,114 parameters
cnn_overparam        35,106,946 parameters


# CELL 12 — TRAINING CONFIGURATION

In [ ]:
# ============================================================
# CELL 12 — TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 1

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0

CRITERION = nn.CrossEntropyLoss()

print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Loss: CrossEntropyLoss")

Epochs: 1
Learning rate: 0.001
Weight decay: 0.0
Loss: CrossEntropyLoss


# CELL 13 — TRAINING FUNCTION

In [ ]:
def train_one_model(
    model,
    model_name,
    epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
):
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    history = []

    print()
    print("=" * 78)
    print(f"TRAINING: {model_name}")
    print("=" * 78)

    total_start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0
        running_correct = 0
        running_total = 0

        epoch_start = time.time()

        for images, labels in train_loader:
            # Direct GPU transfer (avoiding non_blocking async flags on T4 VM)
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).long()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = CRITERION(logits, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = logits.argmax(dim=1)
            running_correct += (predictions == labels).sum().item()
            running_total += images.size(0)

        epoch_loss = running_loss / running_total
        epoch_acc = running_correct / running_total
        epoch_time = time.time() - epoch_start

        history.append({
            "epoch": epoch,
            "loss": epoch_loss,
            "accuracy": epoch_acc,
            "epoch_time_seconds": epoch_time,
        })

        print(
            f"Epoch {epoch:2d}/{epochs} | "
            f"loss={epoch_loss:.6f} | "
            f"accuracy={epoch_acc:.4f} | "
            f"time={epoch_time/60:.2f} min"
        )

    total_time = time.time() - total_start

    print()
    print(f"Finished {model_name} in {total_time/60:.2f} min")

    return model, history

# CELL 14 — CHECKPOINT SAVING

In [ ]:
# ============================================================
# CELL 14 — CHECKPOINT SAVING
# ============================================================

def save_checkpoint(
    model,
    model_name,
    history,
):

    model_path = (
        CHECKPOINT_DIR /
        f"{model_name}.pt"
    )

    metadata_path = (
        CHECKPOINT_DIR /
        f"{model_name}_metadata.json"
    )

    checkpoint = {
        "model_name": model_name,
        "state_dict": model.state_dict(),
        "input_shape": [3, 256, 256],
        "num_classes": 2,
        "task": "CelebA Smiling classification",
        "dataset_split": "train",
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "normalization_mean": [0.5, 0.5, 0.5],
        "normalization_std": [0.5, 0.5, 0.5],
        "history": history,
    }

    torch.save(
        checkpoint,
        model_path,
    )

    metadata = {
        key: value
        for key, value in checkpoint.items()
        if key != "state_dict"
    }

    with open(
        metadata_path,
        "w",
    ) as f:
        json.dump(
            metadata,
            f,
            indent=2,
        )

    print("Saved:")
    print(" ", model_path)
    print(" ", metadata_path)

    return model_path

# CELL 15 — TRAIN ALL FOUR VICTIMS

In [ ]:
# ============================================================
# CELL 15 — SAVE EPOCH 0 & TRAIN EPOCH 1 FOR ALL VICTIMS
# ============================================================

all_histories = {}
trained_models = {}

for model_name, factory in MODEL_FACTORIES.items():

    # 1. Instantiate uninitialized model
    seed_everything(SEED)
    model = factory()

    # 2. Save Epoch 0 (Untrained) Checkpoint
    epoch0_path = CHECKPOINT_DIR / f"{model_name}_epoch0.pt"
    torch.save({
        "model_name": model_name,
        "state_dict": model.state_dict(),
        "epoch": 0,
        "seed": SEED,
    }, epoch0_path)
    print(f"Saved Epoch 0 checkpoint: {epoch0_path.name}")

    # 3. Train for 1 Epoch
    model, history = train_one_model(
        model=model,
        model_name=model_name,
        epochs=NUM_EPOCHS, # = 1
    )

    # 4. Save Epoch 1 Checkpoint
    save_checkpoint(
        model=model,
        model_name=model_name,
        history=history,
    )

    all_histories[model_name] = history
    trained_models[model_name] = model

    print()

Saved Epoch 0 checkpoint: mlp_small_epoch0.pt

TRAINING: mlp_small
Epoch  1/1 | loss=0.544456 | accuracy=0.7338 | time=9.47 min

Finished mlp_small in 9.47 min
Saved:
  /content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/checkpoints/mlp_small.pt
  /content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/checkpoints/mlp_small_metadata.json

Saved Epoch 0 checkpoint: lenet_sigmoid_epoch0.pt

TRAINING: lenet_sigmoid
Epoch  1/1 | loss=0.709862 | accuracy=0.5134 | time=11.62 min

Finished lenet_sigmoid in 11.62 min
Saved:
  /content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/checkpoints/lenet_sigmoid.pt
  /content/drive/MyDrive/GGSS_R_victim_architecture_comparison_for_gradient_inversion_without_diffusion/checkpoints/lenet_sigmoid_metadata.json

Saved Epoch 0 checkpoint: cnn_gi_epoch0.pt

TRAINING: cnn_gi
Epoch  1/1 | loss=0.704550 | accuracy=0.515

# CELL 16 — TRAINING SUMMARY

In [ ]:
# ============================================================
# CELL 16 — TRAINING SUMMARY
# ============================================================

summary_rows = []

for model_name, history in all_histories.items():

    final = history[-1]

    model = trained_models[model_name]

    n_params = sum(
        p.numel()
        for p in model.parameters()
    )

    summary_rows.append({
        "model": model_name,
        "parameters": n_params,
        "final_epoch": final["epoch"],
        "final_train_loss": final["loss"],
        "final_train_accuracy": final["accuracy"],
    })

training_summary = pd.DataFrame(
    summary_rows
)

display(training_summary)

,model,parameters,final_epoch,final_train_loss,final_train_accuracy
0,mlp_small,25166210,1,0.544456,0.733796
1,lenet_sigmoid,34022,1,0.709862,0.513399
2,cnn_gi,4334114,1,0.704550,0.515267
3,cnn_overparam,35106946,1,0.718581,0.506715


# CELL 17 — CHECKPOINT RELOAD TEST

In [ ]:
# ============================================================
# CELL 17 — CHECKPOINT RELOAD TEST
# ============================================================

for model_name, factory in MODEL_FACTORIES.items():

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{model_name}.pt"
    )

    assert checkpoint_path.exists()

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    test_model = factory()

    test_model.load_state_dict(
        checkpoint["state_dict"]
    )

    test_model.eval()

    print(
        f"{model_name:20s} "
        f"✓ checkpoint reload successful"
    )

mlp_small            ✓ checkpoint reload successful
lenet_sigmoid        ✓ checkpoint reload successful
cnn_gi               ✓ checkpoint reload successful
cnn_overparam        ✓ checkpoint reload successful


In [ ]:
import hashlib, torch

path = CHECKPOINT_DIR / "cnn_overparam.pt"

# 1. Verify zip header and load dictionary
ckpt = torch.load(path, map_location="cpu")
state_dict = ckpt["state_dict"]

# 2. Verify tensor parameters exist, are non-empty, and contain no NaNs/Infs
nan_count = sum(torch.isnan(v).sum().item() for v in state_dict.values())
param_count = sum(v.numel() for v in state_dict.values())
md5 = hashlib.md5(open(path, 'rb').read()).hexdigest()

print(f"File Size: {path.stat().st_size / (1024**2):.2f} MB")
print(f"File MD5 Checksum: {md5}")
print(f"Total Parameters Loaded: {param_count:,}")
print(f"NaN or Inf Values Found: {nan_count}")

assert nan_count == 0, "Corrupted weights detected!"
print("\n RESULT: Model file is completely healthy and uncorrupted.")

File Size: 133.93 MB
File MD5 Checksum: a7c5abfe0cafafb4f0ac6221c1b6dbe7
Total Parameters Loaded: 35,106,946
NaN or Inf Values Found: 0

 RESULT: Model file is completely healthy and uncorrupted.
